In [57]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [52]:
import chromadb

client = chromadb.PersistentClient(path="/news/chroma_db/")

collection = client.get_collection('bbc_news')

In [ ]:
# from langchain_huggingface import HuggingFaceEmbeddings

# hg_embeddings = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L6-v2"
# )

d:\AI Projects\RAG Practice\BBC_News_Extraction\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2185.55it/s]


In [66]:
def retrieve_context(query):
    query_embedding = hg_embeddings.embed_query(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3,
        include=["documents", "embeddings", "distances"]
    )
    print(results["distances"][0])

    return results["documents"][0]

In [69]:
def call_llm_with_rag(query):
    chunks = retrieve_context(query)
    context = "\n\n".join(chunks)

    print(context)

    # return ""
    API_KEY = os.getenv('REQUESTY_API_KEY')

    client = OpenAI(
        base_url="https://router.requesty.ai/v1",
        api_key=API_KEY
    )
    
    system_prompt = f"""You are a RAG-based assistant.
Answer the user's question using only the information
provided in the retrieved context.
You may synthesize or infer practical conclusions
from statements in the context, but do not introduce
facts that are not supported by the context.
If the context truly does not contain enough information
to answer the question, say so.
"""

    user_prompt = f"""Context: {context}\n
Question: {query}
"""
 
    response = client.chat.completions.create(
        model="nvidia/nemotron-3-ultra-550b-a55b",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    answer = response.choices[0].message.content
    return answer

In [ ]:
query = input()  # Any news about olympics? If so, which sports are discussed about?
answer = call_llm_with_rag(query)

[1.0499906539916992, 1.0639318227767944, 1.1117264032363892]
isinbayeva heads for birmingham olympic pole vault champion yelena isinbayeva has confirmed she will take part in the norwich union grand prix in birmingham on february everybody knows how much i enjoy competing in britain i always seem to break records there said isinbayeva as olympic champion there will be more attention on me this year but hopefully i can respond with another record in birmingham kelly holmes and carolina kluft are among other athens winners competing the organisers are hoping that isinbayevas main rival fellow russian svetlana feofanova will also take part in the event the pair had a thrilling battle in athens which ended with isinbayeva finally jumping a world record of m to claim the gold medal isinbayeva has set world records in the pole vault three of which have come on british soil

bekele sets sights on world mark olympic m champion kenenisa bekele is determined to add the world indoor two mile reco

"Yes, the context contains news about several Olympic athletes and their achievements across multiple sports:\n\n**Sports mentioned:**\n1. **Pole vault** - Yelena Isinbayeva (Olympic champion, world record holder)\n2. **Middle-distance running (800m & 1500m)** - Kelly Holmes (double Olympic gold medalist in Athens)\n3. **Heptathlon** - Carolina Kluft (Olympic gold medallist)\n4. **Long-distance running (10,000m/two mile)** - Kenenisa Bekele (Olympic 10,000m champion) and Haile Gebrselassie\n5. **Sprint relay (4x100m)** - Jason Gardener and Mark Lewis-Francis (Olympic gold medallists)\n6. **Decathlon** - Daley Thompson (double Olympic champion, mentioned as comparison)\n\nThe articles appear to be from early 2005, discussing these athletes' plans for upcoming indoor meets (Norwich Union Grand Prix in Birmingham, European Indoor Championships) following the 2004 Athens Olympics."

In [71]:
print(answer)

Yes, the context contains news about several Olympic athletes and their achievements across multiple sports:

**Sports mentioned:**
1. **Pole vault** - Yelena Isinbayeva (Olympic champion, world record holder)
2. **Middle-distance running (800m & 1500m)** - Kelly Holmes (double Olympic gold medalist in Athens)
3. **Heptathlon** - Carolina Kluft (Olympic gold medallist)
4. **Long-distance running (10,000m/two mile)** - Kenenisa Bekele (Olympic 10,000m champion) and Haile Gebrselassie
5. **Sprint relay (4x100m)** - Jason Gardener and Mark Lewis-Francis (Olympic gold medallists)
6. **Decathlon** - Daley Thompson (double Olympic champion, mentioned as comparison)

The articles appear to be from early 2005, discussing these athletes' plans for upcoming indoor meets (Norwich Union Grand Prix in Birmingham, European Indoor Championships) following the 2004 Athens Olympics.
